# One zarr with everything: GMST, GWL crossings, regional timeseries

Collapses the scattered CSV/netCDF caches into a single self-describing store,
`summer_data/aux/gwl_timeseries.zarr`, covering **both** archives (RAMIP experiments and the
ScenarioMIP/overshoot download) on one set of coordinates.

```
gwl_timeseries.zarr
├── gmst      gmst_anom(model, exp, member, year), run_id(model, exp, member)
├── gwl       window_end / down_end / peak (model, exp, member, gwl)
└── regional  tas, pr, tasmax, tasmin (model, exp, member, year, region)
```

Design choices worth knowing before using it:

- **`member` is an index, not a label.** Run identifiers are not comparable across models
  (`r101i1p1f1` in MRI, `r1i1p3f1` in GISS), so the dimension is `member = 0…N-1` and the real
  identifier lives in `run_id(model, exp, member)`. Slicing by member index across models is
  meaningless; slice by model first, then read `run_id`.
- **The array is sparse by construction.** No model runs every experiment, so absent
  combinations are NaN. That is data, not corruption — count with `notnull()`, and check
  `run_id` before trusting a slice.
- **`collection` and `anchor_exp`** mark which archive each experiment came from and which arm
  is the reference, so RAMIP's `ssp370` / `ssp370-LE` split does not have to be rediscovered.
- Everything carries units and provenance attrs. Read them: `xr.open_zarr(P, group='gmst').attrs`.

Rebuilt from the caches written by `ramip_data_layer.ipynb`, `ks_location_scenarios.ipynb` and
this notebook (the ScenarioMIP regional means, which nothing else had computed).

In [1]:
import os, glob, warnings
import numpy as np
import pandas as pd
import xarray as xr
import regionmask

import sys
sys.path.insert(0, '/burg-archive/home/mck2199/summer/bcd_me/code')
sys.path.insert(0, '/burg-archive/home/mck2199/summer/bcd_me/path_independence')
from funcs_support import get_params
import ks_tools as kt

dir_list = get_params()
warnings.filterwarnings('ignore', category=FutureWarning)

RAMIP   = dir_list['raw'] + 'ramip/'
SCEN    = dir_list['raw'] + 'scenariomip/'
DL_DIR  = dir_list['aux'] + 'data_layer/'
REG_DIR = DL_DIR + 'regional/'
SCEN_REG = DL_DIR + 'regional_scenariomip/'
os.makedirs(SCEN_REG, exist_ok=True)

PRODUCT = dir_list['aux'] + 'gwl_timeseries.zarr'
VARS  = ['tas', 'pr', 'tasmax', 'tasmin']
GWLS  = [1.5, 2.0, 2.5, 3.0]
WIN   = kt.WIN
BASE  = (1850, 1900)
STAMP = str(pd.Timestamp.now().date())

base_df = pd.read_csv(dir_list['aux'] + f'diagnostics/ramip_gwl/baseline_{BASE[0]}_{BASE[1]}.csv')
BASELINE = dict(zip(base_df.source, base_df.baseline_K))

# which archive each experiment belongs to, and the reference arm of each collection
SCEN_EXPS = ['ssp126', 'ssp245', 'ssp585', 'ssp534-over']
print('product ->', PRODUCT)

/burg-archive/home/mck2199/summer/.venv312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


product -> /burg-archive/home/mck2199/summer/summer_data/aux/gwl_timeseries.zarr


## 1. ScenarioMIP regional means

The RAMIP archive already has AR6-regional annual means (`data_layer/regional/`); the
ScenarioMIP download does not. Same recipe: 46 AR6 land regions plus a `global-land`
aggregate, area-weighted, one netCDF per (variable, model), cached so a rerun is free.

In [2]:
ar6 = regionmask.defined_regions.ar6.land
land110 = regionmask.defined_regions.natural_earth_v5_0_0.land_110


def region_masks(grid):
    m3 = ar6.mask_3D(grid.lon, grid.lat)
    gl = (land110.mask_3D(grid.lon, grid.lat).isel(region=0).expand_dims(region=1)
          .assign_coords(region=[-1], names=('region', ['global-land']),
                         abbrevs=('region', ['GLB'])))
    return xr.concat([m3, gl], dim='region')


for var in ['tas', 'pr']:
    for mod in sorted(os.listdir(SCEN)):
        out_fn = f'{SCEN_REG}{var}_{mod}.nc'
        if os.path.exists(out_fn):
            print(f'{var} {mod}: cached')
            continue
        exps = [e for e in kt.experiments(SCEN, mod, var) if e in SCEN_EXPS]
        if not exps:
            continue
        fn0 = next(iter(kt.member_files(SCEN, mod, exps[0], var).values()))
        grid = xr.open_zarr(fn0, consolidated=False)
        mask3d = region_masks(grid)
        w = np.cos(np.deg2rad(grid.lat))
        per_exp = []
        for exp in exps:
            per_run = []
            for run, fn in kt.member_files(SCEN, mod, exp, var).items():
                ds = xr.open_zarr(fn, consolidated=False)
                ann = ds[var].groupby('time.year').mean()
                per_run.append(ann.weighted(mask3d * w).mean(('lat', 'lon')).compute()
                               .assign_coords(run=run))
            per_exp.append(xr.concat(per_run, dim='run').assign_coords(exp=exp))
            print(f'  {var} {mod} {exp}: {len(per_run)} runs', flush=True)
        xr.concat(per_exp, dim='exp', join='outer').rename(var).to_netcdf(out_fn)
        print(f'{var} {mod} -> {out_fn}')

tas CESM2: cached
tas CNRM-ESM2-1: cached
tas CanESM5: cached
tas GISS-E2-1-G: cached
tas MIROC6: cached
tas MRI-ESM2-0: cached
tas NorESM2-LM: cached
tas UKESM1-0-LL: cached
pr CESM2: cached
pr CNRM-ESM2-1: cached
pr CanESM5: cached
pr GISS-E2-1-G: cached
pr MIROC6: cached
pr MRI-ESM2-0: cached
pr NorESM2-LM: cached
pr UKESM1-0-LL: cached


## 2. GMST, on shared coordinates

RAMIP and ScenarioMIP GMST tables are concatenated, then reshaped from long format onto
`(model, exp, member, year)`. `run_id` keeps the real member identifier.

In [3]:
gm_r = pd.read_csv(DL_DIR + 'gmst_annual_long.csv').assign(collection='ramip')
gm_s = pd.read_csv(DL_DIR + 'scenariomip_gmst_annual.csv').assign(collection='scenariomip')
gm = pd.concat([gm_r, gm_s], ignore_index=True)

# member index per (model, exp); run_id holds the identifier this index stands for
gm['member'] = gm.groupby(['model', 'exp']).run.transform(
    lambda s: s.map({r: i for i, r in enumerate(sorted(s.unique()))}))

MODELS = sorted(gm.model.unique())
EXPS   = sorted(gm.exp.unique())
YEARS  = np.arange(int(gm.year.min()), int(gm.year.max()) + 1)
NMEM   = int(gm.member.max()) + 1
print(f'{len(MODELS)} models x {len(EXPS)} experiments x {NMEM} members x {len(YEARS)} years')

gmst = (gm.set_index(['model', 'exp', 'member', 'year']).gmst_anom
        .to_xarray().reindex(model=MODELS, exp=EXPS, member=np.arange(NMEM), year=YEARS))

rid = (gm.drop_duplicates(['model', 'exp', 'member']).set_index(['model', 'exp', 'member']).run
       .to_xarray().reindex(model=MODELS, exp=EXPS, member=np.arange(NMEM)).fillna(''))

coll = (gm.drop_duplicates('exp').set_index('exp').collection.to_xarray().reindex(exp=EXPS))

gmst_ds = xr.Dataset({'gmst_anom': gmst, 'run_id': rid.astype('U20'),
                      'collection': coll.astype('U12')})
gmst_ds.gmst_anom.attrs = {
    'long_name': 'annual global-mean surface air temperature anomaly',
    'units': 'K', 'baseline': f'{BASE[0]}-{BASE[1]} mean of the model\'s own CMIP6 historical',
    'note': 'area-weighted; CanESM5-1 uses the CanESM5 baseline (no historical on pangeo)'}
gmst_ds.run_id.attrs = {'long_name': 'CMIP6 member identifier for this member index',
                        'note': 'empty string where the (model, exp, member) does not exist'}
gmst_ds.collection.attrs = {'long_name': 'source archive of the experiment'}
print(f'gmst filled: {int(gmst.notnull().sum())} of {gmst.size} slots '
      f'({100 * float(gmst.notnull().mean()):.1f}% — the rest are model/experiment '
      f'combinations that do not exist)')

10 models x 15 experiments x 10 members x 86 years


gmst filled: 47167 of 129000 slots (36.6% — the rest are model/experiment combinations that do not exist)


## 3. GWL crossings

Rising-branch crossing (`window_end`, the first 10-yr window reaching the level) for every
experiment, plus the declining-branch window (`down_end`) and the trajectory `peak`, which only
mean something for the overshoot pathway. Recomputed here from the assembled GMST so the store is
internally consistent rather than inheriting two differently-derived CSVs.

In [4]:
rows = []
for (mod, exp, mem), s in gm.groupby(['model', 'exp', 'member']):
    s = s.sort_values('year')
    yr, va = s.year.values, s.gmst_anom.values
    peak = pd.Series(va, index=yr).rolling(WIN).mean().max()
    for g in GWLS:
        up = kt.gwl_window(yr, va, g, win=WIN)
        dn = kt.window_at_level(yr, va, g, tol=0.1, win=WIN, side='declining')
        rows.append({'model': mod, 'exp': exp, 'member': mem, 'gwl': g,
                     'window_end': up if up is not None else np.nan,
                     'down_end': dn if dn is not None else np.nan,
                     'peak': peak})
cr = pd.DataFrame(rows).set_index(['model', 'exp', 'member', 'gwl']).to_xarray()
cr = cr.reindex(model=MODELS, exp=EXPS, member=np.arange(NMEM), gwl=GWLS)

cr.window_end.attrs = {
    'long_name': f'last year of the first {WIN}-yr running-mean window reaching the GWL',
    'units': 'year', 'note': f'the sample is window_end-{WIN - 1} … window_end inclusive; '
                             'NaN where the run never reaches the level in its record'}
cr.down_end.attrs = {
    'long_name': f'last year of the {WIN}-yr window closest to the GWL on the declining branch',
    'units': 'year', 'tolerance_K': 0.1,
    'note': 'overshoot pathways only; NaN when the run never returns to the level (the trap that '
            'makes a naive "last window above the level" comparison meaningless)'}
cr.peak.attrs = {'long_name': f'maximum {WIN}-yr running-mean GMST anomaly of the run', 'units': 'K'}

print('runs reaching each GWL, by collection:')
print(cr.window_end.notnull().sum(['model', 'member']).to_pandas().astype(int).to_string())

runs reaching each GWL, by collection:
gwl                1.5  2.0  2.5  3.0
exp                                  
ssp126              51   34   20    7
ssp245              53   48   45   37
ssp370              77   70   63   57
ssp370-126aer       90   86   60   59
ssp370-AFR126aer    89   62   56   49
ssp370-ASIA126aer   26   16    6    6
ssp370-EAS126aer    89   71   49   40
ssp370-LE           10   10   10   10
ssp370-NAE126aer    89   70   48   35
ssp370-SAF126ca     20   10    3    1
ssp370-SAS126aer    88   66   44   39
ssp370-SAS126ca     21   12    2    2
ssp370-ramip         2    0    0    0
ssp534-over         27   26   20   10
ssp585              50   50   50   50


## 4. Regional timeseries

Both archives' regional means stacked onto the same `(model, exp, member, year, region)` grid.
`region` is the AR6 land region number with `-1` for the global-land aggregate; `abbrevs` and
`names` come along as coordinates.

In [5]:
KEEP = {'exp', 'run', 'member', 'year', 'region', 'abbrevs', 'names', 'model'}


def regional_da(fn, var):
    """scalar coords like `height` differ between models (2 m vs 1.5 m) and make the
    cross-model concat raise; they carry nothing this product needs"""
    da = xr.open_dataset(fn)[var]
    da = da.drop_vars([c for c in da.coords if c not in KEEP])
    if 'exp' not in da.dims:
        da = da.expand_dims('exp')
    return da


per_var = {}
for var in VARS:
    # a model appears in both regional directories (its RAMIP arms and its SSPs); those are
    # different experiments of the same model, so they merge along `exp`, not along `model`
    by_model = {}
    for fn in sorted(glob.glob(f'{REG_DIR}{var}_*.nc')) + sorted(glob.glob(f'{SCEN_REG}{var}_*.nc')):
        mod = os.path.basename(fn)[len(var) + 1:-3]
        da = regional_da(fn, var)
        # runs -> member index, matching the gmst convention for this (model, exp)
        for exp in da.exp.values:
            d = da.sel(exp=exp).dropna('run', how='all')
            runs = sorted([str(r) for r in d.run.values])
            d = d.sel(run=runs).assign_coords(run=np.arange(len(runs))).rename({'run': 'member'})
            by_model.setdefault(mod, []).append(d.assign_coords(exp=str(exp)))
    parts = [xr.concat(v, dim='exp', join='outer').assign_coords(model=m)
             for m, v in sorted(by_model.items())]
    per_var[var] = xr.concat(parts, dim='model', join='outer')
    print(f'{var}: {dict(per_var[var].sizes)}')

reg = xr.Dataset(per_var).reindex(model=MODELS, exp=EXPS, member=np.arange(NMEM), year=YEARS)
for var in VARS:
    src = xr.open_dataset(sorted(glob.glob(f'{REG_DIR}{var}_*.nc'))[0])[var]
    reg[var].attrs = {k: v for k, v in src.attrs.items()
                      if k in ('standard_name', 'long_name', 'units')}
    reg[var].attrs['cell_methods'] = ('area: mean over AR6 land region (cos-lat weighted); '
                                      'time: annual mean of monthly values')
print('\nregional:', dict(reg.sizes))

tas: {'model': 10, 'exp': 15, 'member': 10, 'year': 86, 'region': 47}


pr: {'model': 10, 'exp': 15, 'member': 10, 'year': 86, 'region': 47}


tasmax: {'model': 9, 'exp': 11, 'member': 10, 'year': 86, 'region': 47}


tasmin: {'model': 9, 'exp': 11, 'member': 10, 'year': 86, 'region': 47}



regional: {'year': 86, 'region': 47, 'member': 10, 'exp': 15, 'model': 10}


## 5. Write the store

One zarr, three groups, provenance in the attrs of each. Chunked along `year` and `region` so a
single model/experiment timeseries reads without touching the rest.

In [6]:
PROV = {
    'title': 'RAMIP + ScenarioMIP GMST, GWL crossings and AR6-regional timeseries',
    'created': STAMP,
    'created_by': 'path_independence/build_data_product.ipynb',
    'baseline_period': f'{BASE[0]}-{BASE[1]}',
    'gwl_window_years': WIN,
    'source_ramip': 'summer_data/raw/ramip (monthly Amon zarr, transferred from deela)',
    'source_scenariomip': 'summer_data/raw/scenariomip (pangeo CMIP6 GCS, tas+pr, <=10 members)',
    'member_note': 'member is an index; the identifier is run_id(model, exp, member)',
    'sparsity_note': 'NaN marks model/experiment/member combinations that do not exist',
    'proxies': 'CanESM5 stands in for CanESM5-1 in the ScenarioMIP collection and for its '
               '1850-1900 baseline; EC-Earth3-AerChem has no ScenarioMIP data',
}

if os.path.exists(PRODUCT):
    import shutil
    shutil.rmtree(PRODUCT)

for name, ds in [('gmst', gmst_ds), ('gwl', cr), ('regional', reg)]:
    ds = ds.copy()
    for v in list(ds.variables):
        ds[v].encoding = {}                       # source encodings break the zarr v3 writer
    ds.attrs = {**PROV, 'group': name}
    chunks = {d: s for d, s in ds.sizes.items() if d in ('year', 'region')}
    ds.chunk(chunks).to_zarr(PRODUCT, group=name, mode='a')
    print(f'wrote group {name}: {dict(ds.sizes)}')

print('\n', PRODUCT)
print('size:', round(sum(os.path.getsize(os.path.join(r, f))
                         for r, _, fs in os.walk(PRODUCT) for f in fs) / 1e6, 1), 'MB')

/burg-archive/home/mck2199/summer/.venv312/lib/python3.12/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


wrote group gmst: {'model': 10, 'exp': 15, 'member': 10, 'year': 86}
wrote group gwl: {'model': 10, 'exp': 15, 'member': 10, 'gwl': 4}


/burg-archive/home/mck2199/summer/.venv312/lib/python3.12/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


wrote group regional: {'year': 86, 'region': 47, 'member': 10, 'exp': 15, 'model': 10}

 /burg-archive/home/mck2199/summer/summer_data/aux/gwl_timeseries.zarr
size: 55.1 MB


/burg-archive/home/mck2199/summer/.venv312/lib/python3.12/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


## 6. Verify against the sources

A store that silently disagrees with the caches it came from is worse than no store. Each check
reopens the written zarr and compares against the original file, not against the in-memory
objects above.

In [7]:
g = xr.open_zarr(PRODUCT, group='gmst')
w = xr.open_zarr(PRODUCT, group='gwl')
r = xr.open_zarr(PRODUCT, group='regional')

# 1. GMST round-trips against the source CSV for a random sample of runs
src = pd.concat([pd.read_csv(DL_DIR + 'gmst_annual_long.csv'),
                 pd.read_csv(DL_DIR + 'scenariomip_gmst_annual.csv')], ignore_index=True)
rng = np.random.default_rng(0)
checked = 0
for _, row in src.sample(200, random_state=1).iterrows():
    mem = int(np.argwhere(g.run_id.sel(model=row.model, exp=row.exp).values == row.run)[0][0])
    got = float(g.gmst_anom.sel(model=row.model, exp=row.exp, member=mem, year=row.year))
    assert abs(got - row.gmst_anom) < 1e-9, (row.model, row.exp, row.run, got, row.gmst_anom)
    checked += 1
print(f'GMST: {checked} sampled values match the source CSV exactly')

# 2. the GWL windows agree with the ones the KS analysis actually used
old = pd.read_csv(dir_list['aux'] + 'diagnostics/ramip_gwl/gwl2.0_windows.csv')
old = old[old.usable]
mism = 0
for _, row in old.iterrows():
    for exp_col, exp in [('anchor_end', 'ssp370-LE' if row.model == 'CESM2' else 'ssp370'),
                         ('other_end', 'ssp370-126aer')]:
        ids = g.run_id.sel(model=row.model, exp=exp).values
        mem = int(np.argwhere(ids == row.run)[0][0])
        got = float(w.window_end.sel(model=row.model, exp=exp, member=mem, gwl=2.0))
        mism += (got != row[exp_col])
print(f'GWL2.0 windows: {mism} disagreements with the windows used by the KS notebooks '
      f'({len(old) * 2} checked)')

# 3. regional means round-trip, and the global-land aggregate tracks GMST over land
fn = f'{REG_DIR}tas_MIROC6.nc'
src_r = xr.open_dataset(fn).tas.sel(exp='ssp370')
runs = sorted([str(x) for x in src_r.dropna('run', how='all').run.values])
a = float(src_r.sel(run=runs[0], year=2050, region=1))
b = float(r.tas.sel(model='MIROC6', exp='ssp370', member=0, year=2050, region=1))
assert abs(a - b) < 1e-9, (a, b)
print(f'regional: MIROC6 ssp370 member 0 region 1 in 2050 matches source ({a:.3f} K)')

# 4. sparsity is where it should be: EC-Earth has no ScenarioMIP, CanESM5 no RAMIP
assert bool(g.gmst_anom.sel(model='EC-Earth3-AerChem', exp='ssp585').isnull().all())
assert bool(g.gmst_anom.sel(model='CanESM5', exp='ssp370-126aer').isnull().all())
assert bool(g.gmst_anom.sel(model='MIROC6', exp='ssp370').notnull().any())
print('sparsity: absent combinations are NaN, present ones are populated')

GMST: 200 sampled values match the source CSV exactly


GWL2.0 windows: 0 disagreements with the windows used by the KS notebooks (144 checked)
regional: MIROC6 ssp370 member 0 region 1 in 2050 matches source (272.892 K)
sparsity: absent combinations are NaN, present ones are populated


In [8]:
print(xr.open_zarr(PRODUCT, group='gmst'))
print()
print('experiments by collection:')
gg = xr.open_zarr(PRODUCT, group='gmst')
for c in np.unique(gg.collection.values):
    print(f'  {c}: {sorted(gg.exp.values[gg.collection.values == c])}')

<xarray.Dataset> Size: 1MB
Dimensions:     (exp: 15, model: 10, member: 10, year: 86)
Coordinates:
  * exp         (exp) object 120B 'ssp126' 'ssp245' ... 'ssp534-over' 'ssp585'
  * model       (model) object 80B 'CESM2' 'CNRM-ESM2-1' ... 'UKESM1-0-LL'
  * member      (member) int64 80B 0 1 2 3 4 5 6 7 8 9
  * year        (year) int64 688B 2015 2016 2017 2018 ... 2097 2098 2099 2100
Data variables:
    collection  (exp) <U12 720B dask.array<chunksize=(15,), meta=np.ndarray>
    gmst_anom   (model, exp, member, year) float64 1MB dask.array<chunksize=(10, 15, 10, 86), meta=np.ndarray>
    run_id      (model, exp, member) <U20 120kB dask.array<chunksize=(10, 15, 10), meta=np.ndarray>
Attributes:
    title:               RAMIP + ScenarioMIP GMST, GWL crossings and AR6-regi...
    created:             2026-08-12
    created_by:          path_independence/build_data_product.ipynb
    baseline_period:     1850-1900
    gwl_window_years:    10
    source_ramip:        summer_data/raw/ramip (mo

## How to use it

```python
import xarray as xr
P = 'summer_data/aux/gwl_timeseries.zarr'

gmst = xr.open_zarr(P, group='gmst')        # gmst_anom(model, exp, member, year)
gwl  = xr.open_zarr(P, group='gwl')         # window_end / down_end / peak
reg  = xr.open_zarr(P, group='regional')    # tas, pr, tasmax, tasmin over AR6 regions

# every run's GWL2.0 decade under the aerosol-cleanup arm
gwl.window_end.sel(exp='ssp370-126aer', gwl=2.0).dropna('member', how='all')

# South Asia annual tas, one model, both global arms
reg.tas.sel(model='MIROC6', region=36, exp=['ssp370', 'ssp370-126aer'])

# which member index is r5i1p1f1?
int((gmst.run_id.sel(model='MIROC6', exp='ssp370') == 'r5i1p1f1').argmax())
```

Not in this store: the pixel-wise KS p-value maps, which live on each model's native grid and
cannot share coordinates without regridding — they stay as per-model netCDFs under
`aux/diagnostics/`.